# Notebook 05: Conference to OpenAlex Source ID Resolution

**Purpose:** Resolve each of the 30 conference acronyms in `huang_matched_openalex.csv` 
to a valid OpenAlex Source ID, so we can later pull ALL papers (not just award winners) 
per (conference, year) for building the non-winning control pool.

**Caching:** This notebook caches API results to `output/nb05_source_search_cache.json`. 
If you've already run this once, re-running will load from cache instantly instead of 
re-querying the OpenAlex API.


In [2]:
import pandas as pd
import os, json, time, requests
from pathlib import Path

DATA_DIR = Path('../conf_data')
FIG_DIR = Path('../conf_figures')

df = pd.read_csv(DATA_DIR / 'huang_matched_openalex.csv')
print(df.shape)
print("Year range:", df['year'].min(), "-", df['year'].max())
print("Conferences:", sorted(df['conference'].dropna().unique()))

(925, 10)
Year range: 2000 - 2018
Conferences: ['AAAI', 'ACL', 'CHI', 'CIKM', 'CVPR', 'FOCS', 'FSE', 'ICCV', 'ICML', 'ICSE', 'ICWSM', 'IJCAI', 'INFOCOM', 'JCDL', 'KDD', 'MOBICOM', 'NeurIPS', 'OSDI', 'PLDI', 'PODS', 'S&P', 'SIGCOMM', 'SIGIR', 'SIGMETRICS', 'SIGMOD', 'SODA', 'SOSP', 'STOC', 'UIST', 'VLDB', 'WWW']


In [3]:
CONF_NAME_MAP = {
    'AAAI': 'AAAI Conference on Artificial Intelligence',
    'ACL': 'Annual Meeting of the Association for Computational Linguistics',
    'CHI': 'Conference on Human Factors in Computing Systems',
    'CIKM': 'Conference on Information and Knowledge Management',
    'CVPR': 'Conference on Computer Vision and Pattern Recognition',
    'FOCS': 'Symposium on Foundations of Computer Science',
    'FSE': 'International Symposium on Foundations of Software Engineering',
    'ICCV': 'International Conference on Computer Vision',
    'ICML': 'International Conference on Machine Learning',
    'ICSE': 'International Conference on Software Engineering',
    'ICWSM': 'International AAAI Conference on Web and Social Media',
    'IJCAI': 'International Joint Conference on Artificial Intelligence',
    'INFOCOM': 'IEEE INFOCOM',
    'JCDL': 'ACM/IEEE Joint Conference on Digital Libraries',
    'KDD': 'Knowledge Discovery and Data Mining',
    'MOBICOM': 'International Conference on Mobile Computing and Networking',
    'NeurIPS': 'Neural Information Processing Systems',
    'OSDI': 'Operating Systems Design and Implementation',
    'PLDI': 'Programming Language Design and Implementation',
    'PODS': 'Symposium on Principles of Database Systems',
    'S&P': 'IEEE Symposium on Security and Privacy',
    'SIGCOMM': 'ACM Special Interest Group on Data Communication',
    'SIGIR': 'International ACM SIGIR Conference',
    'SIGMETRICS': 'ACM SIGMETRICS Conference',
    'SIGMOD': 'ACM SIGMOD International Conference on Management of Data',
    'SODA': 'Symposium on Discrete Algorithms',
    'SOSP': 'Symposium on Operating Systems Principles',
    'STOC': 'Symposium on Theory of Computing',
    'UIST': 'Symposium on User Interface Software and Technology',
    'VLDB': 'Very Large Data Bases',
    'WWW': 'International World Wide Web Conference'
}
print(len(CONF_NAME_MAP), "conferences mapped to search queries")

31 conferences mapped to search queries


In [4]:
CACHE_FILE = 'output/nb05_source_search_cache.json'
os.makedirs('output', exist_ok=True)

if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        search_cache = json.load(f)
    print(f"CACHE HIT: Loaded {len(search_cache)}/{len(CONF_NAME_MAP)} conferences — no API calls needed")
else:
    search_cache = {}
    print("No cache found, will query OpenAlex API")

No cache found, will query OpenAlex API


In [5]:
for conf, query in CONF_NAME_MAP.items():
    if conf in search_cache:
        continue
    try:
        r = requests.get("https://api.openalex.org/sources", params={
            'search': query,
            'filter': 'type:conference',
            'per-page': 5
        }, timeout=20)
        r.raise_for_status()
        data = r.json()
        top_results = [
            {'id': res['id'], 'display_name': res['display_name'],
             'works_count': res.get('works_count'), 'type': 'conference'}
            for res in data.get('results', [])
        ]
        search_cache[conf] = top_results
        top = top_results[0]['display_name'] if top_results else "NO MATCH"
        print(f"{conf} -> {top}")
    except Exception as e:
        print(conf, "ERROR", repr(e))
        search_cache[conf] = []
    with open(CACHE_FILE, 'w') as f:
        json.dump(search_cache, f, indent=2)
    time.sleep(0.3)

print("\nDone. Total cached:", len(search_cache))

AAAI -> Proceedings of the AAAI Conference on Artificial Intelligence
ACL -> Proceedings of the 60th Annual Meeting of the Association for Computational Linguistics (Volume 1: Long Papers)
CHI -> CHI Conference on Human Factors in Computing Systems
CIKM -> Conference on Information and Knowledge Management
CVPR -> 2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
FOCS -> 2022 IEEE 63rd Annual Symposium on Foundations of Computer Science (FOCS)
FSE -> NO MATCH
ICCV -> 2021 IEEE/CVF International Conference on Computer Vision (ICCV)
ICML -> International Conference on Machine Learning
ICSE -> International Conference on Software Engineering
ICWSM -> Proceedings of the International AAAI Conference on Web and Social Media
IJCAI -> International Joint Conference on Artificial Intelligence
INFOCOM -> IEEE INFOCOM 2022 - IEEE Conference on Computer Communications
JCDL -> ACM/IEEE Joint Conference on Digital Libraries
KDD -> Knowledge Discovery and Data Mining
MOBICOM

## Build Final Resolution Table

Known caveats to verify manually before Notebook 02:
- **CVPR, ICCV, FOCS, INFOCOM, S&P**: OpenAlex may split these into year-specific Source IDs 
  (e.g. "2022 IEEE/CVF CVPR") rather than one stable umbrella ID. Since our data spans 2000-2018, 
  we may need name+year filtering instead of a single source_id for these.
- **WWW**: top match may be a side-track proceedings, not the main venue — needs a corrected query.


In [6]:
final_resolution = {}
for conf, results in search_cache.items():
    if not results:
        final_resolution[conf] = {'source_id': None, 'source_name': None, 'status': 'UNRESOLVED'}
        continue
    best = results[0]
    final_resolution[conf] = {'source_id': best['id'], 'source_name': best['display_name'], 'status': 'RESOLVED'}

res_df = pd.DataFrame(final_resolution).T.reset_index().rename(columns={'index': 'conference'})
res_df.to_csv('output/nb05_conference_source_ids.csv', index=False)
print(res_df['status'].value_counts())
res_df

status
RESOLVED      28
UNRESOLVED     3
Name: count, dtype: int64


,conference,source_id,source_name,status
0,AAAI,https://openalex.org/S4210191458,Proceedings of the AAAI Conference on Artifici...,RESOLVED
1,ACL,https://openalex.org/S4363608652,Proceedings of the 60th Annual Meeting of the ...,RESOLVED
2,CHI,https://openalex.org/S4363607743,CHI Conference on Human Factors in Computing S...,RESOLVED
3,CIKM,https://openalex.org/S4306418063,Conference on Information and Knowledge Manage...,RESOLVED
4,CVPR,https://openalex.org/S4363607701,2022 IEEE/CVF Conference on Computer Vision an...,RESOLVED
5,FOCS,https://openalex.org/S4363607389,2022 IEEE 63rd Annual Symposium on Foundations...,RESOLVED
6,FSE,None,None,UNRESOLVED
7,ICCV,https://openalex.org/S4363607764,2021 IEEE/CVF International Conference on Comp...,RESOLVED
8,ICML,https://openalex.org/S4306419644,International Conference on Machine Learning,RESOLVED
9,ICSE,https://openalex.org/S4306419842,International Conference on Software Engineering,RESOLVED
